# 📄 Research Paper Question-Answering System (RAG)

**Problem 2 — Build a RAG application that lets users upload research papers and ask questions about their contents.**

This notebook implements the full pipeline *from scratch* (no black-box LangChain chains) so that every step is transparent and defensible in a viva:

1. **PDF ingestion** — upload one or more papers
2. **Text extraction** — page-aware extraction with PyMuPDF
3. **Chunking** — word-based sliding window with overlap, tunable
4. **Embedding + Semantic retrieval** — Sentence-Transformers + FAISS (cosine similarity)
5. **Context-grounded generation** — LLM only answers from retrieved chunks, refuses to hallucinate beyond them
6. **Source citation** — every answer is traced back to `(paper, page range, chunk id)`

At the end there's a **Viva Prep** section with talking points and a live experiment cell comparing chunk sizes and retrieval depth.

---
### How to use this notebook
1. Runtime → Change runtime type → (CPU is fine, GPU is optional/faster for embeddings)
2. Run cells top to bottom
3. When prompted, upload your PDF paper(s)
4. Get a free Gemini API key at https://aistudio.google.com/apikey (or use OpenAI — see the config cell)
5. Ask questions in the Q&A cell at the bottom


## 1. Install dependencies

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu google-generativeai openai tiktoken
print("✅ Dependencies installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 56.0 MB/s eta 0:00:00
✅ Dependencies installed


## 2. Configuration

These are the knobs the viva panel will likely ask you to justify. Defaults below are sensible starting points — the **Viva Prep** section at the end explains *why*, and lets you re-run with different values to show the effect live.

| Parameter | Default | What it controls |
|---|---|---|
| `CHUNK_SIZE_WORDS` | 300 | Size of each retrieval unit (~350-450 tokens). Small = precise but loses surrounding context. Large = more context but noisier embeddings / dilutes relevance. |
| `CHUNK_OVERLAP_WORDS` | 50 (~17%) | Prevents a sentence/idea that straddles a chunk boundary from being lost entirely in either chunk. |
| `TOP_K` | 4 | Retrieval depth — how many chunks are fed to the LLM as context. Too low → misses relevant evidence. Too high → dilutes the prompt with irrelevant text and raises hallucination/cost risk. |
| `EMBED_MODEL` | all-MiniLM-L6-v2 | Fast, 384-dim, strong general-purpose sentence embedding model — good CPU speed/quality trade-off for a Colab demo. |
| `LLM_PROVIDER` | gemini | Swap to "openai" if you have an OpenAI key instead. |


AQ.Ab8RN6LaxwaXlygdaUH9P9FXFS51hyTyUf2SDIHF84DUyeth3A

In [ ]:
# ---- Retrieval / chunking configuration ----
CHUNK_SIZE_WORDS   = 300
CHUNK_OVERLAP_WORDS = 50
TOP_K              = 4
EMBED_MODEL_NAME   = "all-MiniLM-L6-v2"

# ---- LLM provider: "gemini" or "openai" ----
LLM_PROVIDER = "gemini"

import getpass, os

if LLM_PROVIDER == "gemini":
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key (get one free at https://aistudio.google.com/apikey): ")
elif LLM_PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

print(f"✅ Configured provider: {LLM_PROVIDER}")


Enter your Gemini API key (get one free at https://aistudio.google.com/apikey): ··········
✅ Configured provider: gemini


## 3. PDF ingestion

Upload one or more research papers (PDF). Each uploaded file is tracked separately so citations can say *which* paper an answer came from — useful if you want to demo multi-paper QA.

In [ ]:
from google.colab import files
import os

UPLOAD_DIR = "/content/papers"
os.makedirs(UPLOAD_DIR, exist_ok=True)

uploaded = files.upload()  # opens a file picker — select one or more PDFs

paper_paths = []
for fname, content in uploaded.items():
    path = os.path.join(UPLOAD_DIR, fname)
    with open(path, "wb") as f:
        f.write(content)
    paper_paths.append(path)

print(f"✅ Uploaded {len(paper_paths)} paper(s):")
for p in paper_paths:
    print(" -", os.path.basename(p))


Saving bert.pdf to bert.pdf
✅ Uploaded 1 paper(s):
 - bert.pdf


## 4. Text extraction

We extract text **page by page** (not the whole PDF as one blob) so that every chunk can retain a `page_start`/`page_end` for citation later.

In [ ]:
import fitz  # PyMuPDF

def extract_pages(pdf_path):
    """Return a list of (page_number, text) tuples, 1-indexed."""
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text("text")
        pages.append((i + 1, text))
    doc.close()
    return pages

papers_pages = {}  # paper filename -> list of (page_num, text)
for path in paper_paths:
    name = os.path.basename(path)
    pages = extract_pages(path)
    papers_pages[name] = pages
    total_words = sum(len(t.split()) for _, t in pages)
    print(f"{name}: {len(pages)} pages, ~{total_words} words extracted")


bert.pdf: 16 pages, ~10152 words extracted


## 5. Chunking (sliding window with overlap)

We chunk on **word count**, sliding the window forward by `CHUNK_SIZE_WORDS - CHUNK_OVERLAP_WORDS` each step. This is simple, predictable, and easy to defend in a viva compared to a black-box splitter. Each chunk keeps:
- the source paper
- the page range it was drawn from (approximated by tracking word→page boundaries)
- a chunk id

> **Why overlap matters:** without it, a sentence like *"...the model achieves 92% accuracy [end of chunk] on the held-out test set..."* could get split so neither half retrieves well. Overlap re-includes the boundary text in both neighboring chunks.

In [ ]:
import json

def chunk_paper(pages, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    """
    pages: list of (page_num, text)
    Returns list of dicts: {text, page_start, page_end, chunk_id}
    """
    # Flatten into a single word stream, remembering which page each word came from
    word_page_pairs = []
    for page_num, text in pages:
        for w in text.split():
            word_page_pairs.append((w, page_num))

    chunks = []
    step = max(chunk_size - overlap, 1)
    i = 0
    chunk_id = 0
    while i < len(word_page_pairs):
        window = word_page_pairs[i:i + chunk_size]
        if not window:
            break
        words = [w for w, _ in window]
        page_nums = [p for _, p in window]
        chunks.append({
            "chunk_id": chunk_id,
            "text": " ".join(words),
            "page_start": min(page_nums),
            "page_end": max(page_nums),
        })
        chunk_id += 1
        i += step
    return chunks

all_chunks = []  # each entry also carries its source paper name
for paper_name, pages in papers_pages.items():
    chunks = chunk_paper(pages)
    for c in chunks:
        c["paper"] = paper_name
        all_chunks.append(c)

print(f"✅ Built {len(all_chunks)} chunks across {len(papers_pages)} paper(s)")
print("\nSample chunk:")
print(json.dumps({k: (v[:200] + '...' if k == 'text' else v) for k, v in all_chunks[0].items()}, indent=2))


✅ Built 41 chunks across 1 paper(s)

Sample chunk:
{
  "chunk_id": 0,
  "text": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova Google AI Language {jacobdevlin,mingweichang,kentonl,kristout...",
  "page_start": 1,
  "page_end": 1,
  "paper": "bert.pdf"
}


## 6. Embeddings + semantic index (FAISS)

Each chunk is embedded with a Sentence-Transformers model, normalized, and stored in a FAISS **inner-product** index (equivalent to cosine similarity on normalized vectors). This is the "semantic retrieval" component — it retrieves by *meaning*, not keyword overlap, so a question phrased differently from the paper's wording can still find the right chunk.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embed_model = SentenceTransformer(EMBED_MODEL_NAME)

def build_index(chunks):
    texts = [c["text"] for c in chunks]
    embeddings = embed_model.encode(texts, convert_to_numpy=True, show_progress_bar=True, normalize_embeddings=True)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # inner product on normalized vecs = cosine similarity
    index.add(embeddings.astype("float32"))
    return index, embeddings

faiss_index, chunk_embeddings = build_index(all_chunks)
print(f"✅ Indexed {faiss_index.ntotal} chunks, embedding dim = {chunk_embeddings.shape[1]}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Indexed 41 chunks, embedding dim = 384


## 7. Retrieval

In [ ]:
def retrieve(query, k=TOP_K, index=None, chunks=None):
    index = index or faiss_index
    chunks = chunks or all_chunks
    q_emb = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        c = dict(chunks[idx])
        c["score"] = float(score)
        results.append(c)
    return results

# quick sanity check
test_results = retrieve("What is the main objective of this paper?")
for r in test_results:
    print(f"[{r['paper']} p.{r['page_start']}-{r['page_end']}] score={r['score']:.3f}")
    print(" ", r["text"][:150].replace(chr(10), " "), "...\n")


[bert.pdf p.15-15] score=0.239
  Tok N Tok 1 ... Tok M Question Paragraph BERT E[CLS] E1 E2 EN C T1 T2 TN Single Sentence ... ... BERT Tok 1 Tok 2 Tok N ... [CLS] E[CLS] E1 E2 EN C T1 ...

[bert.pdf p.12-12] score=0.203
  arXiv:1609.08144. Jason Yosinski, Jeff Clune, Yoshua Bengio, and Hod Lipson. 2014. How transferable are features in deep neural networks? In Advances  ...

[bert.pdf p.11-11] score=0.184
  models. In ACL. Matthew Peters, Mark Neumann, Mohit Iyyer, Matt Gardner, Christopher Clark, Kenton Lee, and Luke Zettlemoyer. 2018a. Deep contextualiz ...

[bert.pdf p.4-5] score=0.181
  tionships, we pre-train for a binarized next sen- tence prediction task that can be trivially gener- ated from any monolingual corpus. Speciﬁcally, wh ...



## 8. Context-grounded generation + citation

The prompt explicitly instructs the model to:
- answer **only** from the provided context chunks
- say so if the answer isn't in the retrieved context (instead of guessing)
- cite which source chunk(s) each claim comes from

This is the core of what makes it "RAG" rather than "an LLM with some text pasted in" — the grounding instruction plus retrieval is what keeps answers tied to the actual paper.

In [ ]:
def build_prompt(question, retrieved_chunks):
    context_blocks = []
    for i, c in enumerate(retrieved_chunks, start=1):
        tag = f"[Source {i}: {c['paper']}, p.{c['page_start']}-{c['page_end']}]"
        context_blocks.append(f"{tag}\n{c['text']}")
    context = "\n\n".join(context_blocks)

    prompt = f"""You are a research assistant answering questions ONLY using the provided paper excerpts.

Rules:
- Answer strictly from the CONTEXT below. Do not use outside knowledge.
- If the context does not contain enough information to answer, say: "The paper does not appear to address this."
- After each claim, cite the source using its bracket tag, e.g. [Source 2].
- Be concise and precise (a few sentences unless the question needs a list).

CONTEXT:
{context}

QUESTION: {question}

ANSWER (with citations):"""
    return prompt


def call_llm(prompt, max_retries=5):
    import time
    for attempt in range(max_retries):
        try:
            if LLM_PROVIDER == "gemini":
                import google.generativeai as genai
                genai.configure(api_key=os.environ["GEMINI_API_KEY"])
                model = genai.GenerativeModel("gemini-3.6-flash")
                resp = model.generate_content(prompt)
                return resp.text
            elif LLM_PROVIDER == "openai":
                from openai import OpenAI
                client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
                resp = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.2,
                )
                return resp.choices[0].message.content
            else:
                raise ValueError("Unknown LLM_PROVIDER")
        except Exception as e:
            err_str = str(e)
            is_rate_limit = "429" in err_str or "TooManyRequests" in err_str or "quota" in err_str.lower()
            if is_rate_limit and attempt < max_retries - 1:
                wait_s = 20 * (attempt + 1)
                print(f"  ⏳ Rate limited, waiting {wait_s}s before retry {attempt + 2}/{max_retries}...")
                time.sleep(wait_s)
                continue
            raise


def answer_question(question, k=TOP_K, verbose_sources=True):
    retrieved = retrieve(question, k=k)
    prompt = build_prompt(question, retrieved)
    answer = call_llm(prompt)

    print(f"Q: {question}\n")
    print(f"A: {answer}\n")
    if verbose_sources:
        print("Sources used:")
        for i, c in enumerate(retrieved, start=1):
            print(f"  [Source {i}] {c['paper']}, p.{c['page_start']}-{c['page_end']} (similarity={c['score']:.3f})")
    print("\n" + "-"*80 + "\n")
    return answer, retrieved


## 9. Run the standard question set

These are exactly the question types from the problem statement.

In [ ]:
standard_questions = [
    "What is the objective of the paper?",
    "What methodology was used?",
    "What datasets were used?",
    "What are the major findings?",
    "What are the limitations?",
]

import time as _time
for i, q in enumerate(standard_questions):
    answer_question(q)
    if i < len(standard_questions) - 1:
        _time.sleep(15)


Q: What is the objective of the paper?

A: Based on the provided context, the core argument and objective of the paper is to demonstrate that bi-directionality and the authors' two pre-training tasks account for the majority of empirical improvements in language representation models [Source 2]. To show this, the paper evaluates BERT by fine-tuning it on 11 NLP tasks using minimal task-specific additional parameters learned from scratch [Source 2, Source 4].

Sources used:
  [Source 1] bert.pdf, p.15-15 (similarity=0.223)
  [Source 2] bert.pdf, p.14-14 (similarity=0.191)
  [Source 3] bert.pdf, p.7-7 (similarity=0.185)
  [Source 4] bert.pdf, p.5-6 (similarity=0.182)

--------------------------------------------------------------------------------

Q: What methodology was used?

A: Based on the provided context, the methodology varies between pre-training, fine-tuning, and task-specific implementations:

* **Pre-training:** BERT utilizes a bidirectional architecture trained on BooksCorpu

ERROR:tornado.access:503 POST /v1beta/models/gemini-3.6-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6468.26ms


Q: What are the limitations?

A: The paper does not appear to address this.

Sources used:
  [Source 1] bert.pdf, p.8-9 (similarity=0.197)
  [Source 2] bert.pdf, p.11-11 (similarity=0.161)
  [Source 3] bert.pdf, p.7-7 (similarity=0.148)
  [Source 4] bert.pdf, p.14-14 (similarity=0.148)

--------------------------------------------------------------------------------



## 10. Ask your own questions (interactive)

Run this cell and type questions; type `exit` to stop.

In [ ]:
while True:
    q = input("Ask a question about the paper (or 'exit'): ")
    if q.strip().lower() == "exit":
        break
    answer_question(q)


Q: summmary of it

A: Based on the provided paper excerpts, here is a summary of the main points:

* **Fine-Tuning on Natural Language Tasks:** BERT can be fine-tuned across a variety of single-sentence and sentence-pair tasks, including SST-2, CoLA, STS-B, MRPC, RTE, and WNLI [Source 1]. 
* **State-of-the-Art GLUE Performance:** Evaluated on the GLUE benchmark, both $\text{BERT}_{\text{BASE}}$ and $\text{BERT}_{\text{LARGE}}$ outperform all prior systems by significant margins, achieving average accuracy improvements of 4.5% and 7.0% over the previous state of the art, respectively [Source 3].
* **Versatility with Feature-Based Approaches:** In addition to fine-tuning, BERT is effective when used as a feature extractor without tuning its pre-trained parameters [Source 4]. For example, on the CoNLL-2003 Named Entity Recognition task, concatenating token representations from BERT's top four hidden layers scores within 0.3 F1 of fine-tuning the full model [Source 4].
* **Conclusion:** Th

---
## 🎓 Viva Prep

### Q: Why RAG instead of directly asking an LLM?

- **Grounding / hallucination control**: an LLM asked directly about a paper it hasn't seen will confidently *make up* plausible-sounding methodology, datasets, or numbers. RAG forces the answer to be derived from retrieved text that actually came from the PDF.
- **No fine-tuning needed**: you don't need to (and can't practically) fine-tune a model per uploaded paper. Retrieval lets any general-purpose LLM answer about *any* newly uploaded document at inference time.
- **Papers exceed context economically**: even if a paper fits in a long context window, pasting the *entire* PDF into every prompt is wasteful and slow, and irrelevant sections can distract the model ("lost in the middle" effect). Retrieval selects only the relevant slice.
- **Freshness**: the LLM's training data has a cutoff; a paper uploaded today (or an obscure/unpublished one) was never seen during training. RAG doesn't rely on the model "knowing" the paper.
- **Verifiability**: because answers are tied to specific chunks, you get citations — the user can check the claim against the source page, which a raw LLM answer can't offer.

### Q: How did you choose chunk size and overlap?

- **Chunk size (300 words ≈ 350-450 tokens)**: small enough that each chunk's embedding represents one coherent idea (so semantic search is precise), but large enough to contain full sentences/paragraphs with enough context to be useful standalone. Too small (e.g. 50 words) → embeddings become noisy/ambiguous and you need many more chunks to reconstruct an idea. Too large (e.g. 1500 words) → a chunk's embedding becomes an average over multiple topics, hurting retrieval precision, and wastes prompt budget with irrelevant text.
- **Overlap (50 words, ~17%)**: guards against a key sentence being split across a chunk boundary and therefore under-represented in *both* halves. A common rule of thumb is 10-20% of chunk size — enough to preserve boundary context without duplicating so much content that the index bloats and near-duplicate chunks compete in retrieval.

### Q: How did you choose retrieval depth (top-k)?

- **Too low (k=1)**: a question whose answer spans two paragraphs (e.g., "datasets used" listed with details a paragraph later) may only retrieve part of the answer.
- **Too high (k=15)**: floods the prompt with mostly-irrelevant chunks, increasing cost/latency and — because LLMs weight context unevenly — can actually *reduce* answer quality ("needle in a haystack" dilution) and increases the chance the model pulls an unrelated fact and mis-cites it.
- **k=4 (default here)** balances covering multi-part answers against keeping the context focused. The experiment below lets you show the panel this trade-off directly.

### Live experiment: chunk size & top-k sensitivity

Run this to rebuild the index with different settings and compare retrieval quality side-by-side — useful to show you understand the trade-off, not just picked default numbers.

In [ ]:
def experiment(question, chunk_size, overlap, k):
    exp_chunks = []
    for paper_name, pages in papers_pages.items():
        cs = chunk_paper(pages, chunk_size=chunk_size, overlap=overlap)
        for c in cs:
            c["paper"] = paper_name
            exp_chunks.append(c)
    exp_index, _ = build_index(exp_chunks)
    results = retrieve(question, k=k, index=exp_index, chunks=exp_chunks)

    print(f"--- chunk_size={chunk_size}, overlap={overlap}, top_k={k} -> {len(exp_chunks)} total chunks ---")
    for r in results:
        print(f"  [{r['paper']} p.{r['page_start']}-{r['page_end']}] score={r['score']:.3f}: {r['text'][:120].replace(chr(10),' ')}...")
    print()

q = "What datasets were used in this paper?"

experiment(q, chunk_size=100, overlap=20, k=4)   # small chunks
experiment(q, chunk_size=300, overlap=50, k=4)   # default
experiment(q, chunk_size=800, overlap=100, k=4)  # large chunks
experiment(q, chunk_size=300, overlap=50, k=1)   # shallow retrieval
experiment(q, chunk_size=300, overlap=50, k=8)   # deep retrieval


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

--- chunk_size=100, overlap=20, top_k=4 -> 127 total chunks ---
  [bert.pdf p.15-15] score=0.339: They were annotated with a score from 1 to 5 denoting how similar the two sentences are in terms of semantic meaning. MR...
  [bert.pdf p.10-10] score=0.330: son. 2013. One billion word benchmark for measur- ing progress in statistical language modeling. arXiv preprint arXiv:13...
  [bert.pdf p.5-5] score=0.313: al., 2018a) is a col- lection of diverse natural language understanding tasks. Detailed descriptions of GLUE datasets ar...
  [bert.pdf p.15-15] score=0.307: [CLS] E[CLS] E1 E2 EN C T1 T2 TN Single Sentence B-PER O O ... ... E[CLS] E1 E[SEP] Class Label ... EN E1’ ... EM’ C T1 ...



Batches:   0%|          | 0/2 [00:00<?, ?it/s]

--- chunk_size=300, overlap=50, top_k=4 -> 41 total chunks ---
  [bert.pdf p.7-7] score=0.326: an an- swer span with start and end at the [CLS] to- ken. The probability space for the start and end answer span positi...
  [bert.pdf p.12-12] score=0.310: arXiv:1609.08144. Jason Yosinski, Jeff Clune, Yoshua Bengio, and Hod Lipson. 2014. How transferable are features in deep...
  [bert.pdf p.11-11] score=0.270: models. In ACL. Matthew Peters, Mark Neumann, Mohit Iyyer, Matt Gardner, Christopher Clark, Kenton Lee, and Luke Zettlem...
  [bert.pdf p.9-9] score=0.268: the number of layers; #H = hidden size; #A = number of at- tention heads. “LM (ppl)” is the masked LM perplexity of held...



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

--- chunk_size=800, overlap=100, top_k=4 -> 15 total chunks ---
  [bert.pdf p.9-10] score=0.319: pre-compute an expensive representation of the training data once and then run many experiments with cheaper models on t...
  [bert.pdf p.10-11] score=0.318: Association for Computational Lin- guistics. Ciprian Chelba, Tomas Mikolov, Mike Schuster, Qi Ge, Thorsten Brants, Phill...
  [bert.pdf p.11-12] score=0.270: models. In ACL. Matthew Peters, Mark Neumann, Mohit Iyyer, Matt Gardner, Christopher Clark, Kenton Lee, and Luke Zettlem...
  [bert.pdf p.7-8] score=0.245: 1.1 problem deﬁnition by allowing for the possibility that no short answer exists in the provided para- graph, making th...



Batches:   0%|          | 0/2 [00:00<?, ?it/s]

--- chunk_size=300, overlap=50, top_k=1 -> 41 total chunks ---
  [bert.pdf p.7-7] score=0.326: an an- swer span with start and end at the [CLS] to- ken. The probability space for the start and end answer span positi...



Batches:   0%|          | 0/2 [00:00<?, ?it/s]

--- chunk_size=300, overlap=50, top_k=8 -> 41 total chunks ---
  [bert.pdf p.7-7] score=0.326: an an- swer span with start and end at the [CLS] to- ken. The probability space for the start and end answer span positi...
  [bert.pdf p.12-12] score=0.310: arXiv:1609.08144. Jason Yosinski, Jeff Clune, Yoshua Bengio, and Hod Lipson. 2014. How transferable are features in deep...
  [bert.pdf p.11-11] score=0.270: models. In ACL. Matthew Peters, Mark Neumann, Mohit Iyyer, Matt Gardner, Christopher Clark, Kenton Lee, and Luke Zettlem...
  [bert.pdf p.9-9] score=0.268: the number of layers; #H = hidden size; #A = number of at- tention heads. “LM (ppl)” is the masked LM perplexity of held...
  [bert.pdf p.10-11] score=0.267: pages 3079–3087. J. Deng, W. Dong, R. Socher, L.-J. Li, K. Li, and L. Fei- Fei. 2009. ImageNet: A Large-Scale Hierarchic...
  [bert.pdf p.15-15] score=0.264: Tok N Tok 1 ... Tok M Question Paragraph BERT E[CLS] E1 E2 EN C T1 T2 TN Single Sentence ... ... BERT Tok 1 Tok 2 Tok

### Other likely viva questions

- **"What if the retrieved chunks don't contain the answer?"** → The prompt explicitly instructs the model to say the paper doesn't address it rather than guessing, which you can demo by asking an off-topic question.
- **"How would you scale this to many papers / a paper library?"** → Same FAISS index already supports multi-paper chunks (see the `paper` field in metadata); for large scale you'd move from an in-memory `IndexFlatIP` to an approximate-nearest-neighbor index (`IndexIVFFlat`/HNSW) or a managed vector DB (Pinecone, Weaviate, Chroma) and add metadata filtering (search within one paper only).
- **"Why FAISS and not just keyword search?"** → Keyword/BM25 search fails when the question's wording differs from the paper's wording (e.g., "what data was tested on" vs. paper says "benchmark corpus"). Embedding-based semantic search matches by meaning, not exact tokens. (A stronger production answer: hybrid search — combine BM25 + embeddings — often beats either alone.)
- **"How do you prevent hallucinated citations?"** → Each source tag is generated by *us*, not the model — the model only chooses which tag to reference; the underlying text-to-tag mapping is deterministic code, not an LLM guess.
